Mobilenet model trainign computational cost

In [1]:
import os
import time
import numpy as np
import pandas as pd
import tensorflow as tf
import psutil
from sklearn.model_selection import train_test_split
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam


def memory_usage():
    return psutil.Process().memory_info().rss / (1024 * 1024)


def execution_time(start_time):
    return round(time.time() - start_time, 2)


def count_model_flops(model):
    from tensorflow.python.framework.convert_to_constants import convert_variables_to_constants_v2

    @tf.function(input_signature=[tf.TensorSpec([1, 224, 224, 3], tf.float32)])
    def forward(inputs):
        return model(inputs)

    frozen_function = convert_variables_to_constants_v2(forward.get_concrete_function())
    graph = tf.Graph()
    with graph.as_default():
        tf.graph_util.import_graph_def(frozen_function.graph.as_graph_def(), name='')
    options = tf.compat.v1.profiler.ProfileOptionBuilder.float_operation()
    profile = tf.compat.v1.profiler.profile(graph=graph, options=options)
    return int(profile.total_float_ops)


# BTT dataset
image_dir = '/Users/kaivalyapuranik/Downloads/archive'
class_labels = sorted(
    name for name in os.listdir(image_dir)
    if os.path.isdir(os.path.join(image_dir, name))
)
image_paths = []
labels = []
for label in class_labels:
    class_path = os.path.join(image_dir, label)
    for image_name in os.listdir(class_path):
        if image_name.lower().endswith(('.png', '.jpg', '.jpeg')):
            image_paths.append(os.path.join(class_path, image_name))
            labels.append(label)

image_df = pd.DataFrame({'image_path': image_paths, 'label': labels})
if image_df.empty:
    raise ValueError('BTT dataset is empty.')

train_df, test_df = train_test_split(
    image_df,
    test_size=0.2,
    stratify=image_df['label'],
    random_state=42
)
train_df, val_df = train_test_split(
    train_df,
    test_size=0.2,
    stratify=train_df['label'],
    random_state=42
)

data_gen = ImageDataGenerator(rescale=1.0 / 255.0)
train_gen = data_gen.flow_from_dataframe(
    train_df, x_col='image_path', y_col='label', target_size=(224, 224),
    class_mode='categorical', batch_size=16
)
val_gen = data_gen.flow_from_dataframe(
    val_df, x_col='image_path', y_col='label', target_size=(224, 224),
    class_mode='categorical', batch_size=16
)
test_gen = data_gen.flow_from_dataframe(
    test_df, x_col='image_path', y_col='label', target_size=(224, 224),
    class_mode='categorical', batch_size=16, shuffle=False
)

base_model = MobileNetV2(weights='imagenet', include_top=False, input_shape=(224, 224, 3))
base_model.trainable = False
x = GlobalAveragePooling2D()(base_model.output)
x = Dense(128, activation='relu')(x)
out = Dense(len(class_labels), activation='softmax')(x)
model = Model(inputs=base_model.input, outputs=out)

trainable_params = np.sum([np.prod(variable.shape) for variable in model.trainable_weights])
total_params = np.sum([np.prod(variable.shape) for variable in model.weights])
flops = count_model_flops(model)
print(f'Total Parameters: {total_params:,}')
print(f'Trainable Parameters: {trainable_params:,}')
print(f'MobileNetV2 FLOPs (measured): {flops:,}')

model.compile(optimizer=Adam(learning_rate=0.001), loss='categorical_crossentropy', metrics=['accuracy'])
start_time = time.time()
train_start_memory = memory_usage()
history = model.fit(train_gen, validation_data=val_gen, epochs=10)
train_end_memory = memory_usage()
train_time = execution_time(start_time)

print('\n===== COMPUTATIONAL COST SUMMARY =====')
print(f'Model Training Time: {train_time} seconds')
print(f'Memory Used During Training: {train_end_memory - train_start_memory:.2f} MB')
print(f'Total Model Parameters: {total_params:,}')
print(f'Trainable Parameters: {trainable_params:,}')
print(f'MobileNetV2 FLOPs (measured): {flops:,}')
model.save('mobilenetv2_btt_timing_model.h5')

with open('computational_cost_summary.txt', 'w') as file:
    file.write(f'Model Training Time: {train_time} seconds\n')
    file.write(f'Memory Used During Training: {train_end_memory - train_start_memory:.2f} MB\n')
    file.write(f'Total Model Parameters: {total_params:,}\n')
    file.write(f'Trainable Parameters: {trainable_params:,}\n')
    file.write(f'MobileNetV2 FLOPs (measured): {flops:,}\n')

Found 1536 validated image filenames belonging to 2 classes.
Found 384 validated image filenames belonging to 2 classes.
Found 480 validated image filenames belonging to 2 classes.


I0000 00:00:1790594343.396995  782487 devices.cc:76] Number of eligible GPUs (core count >= 8, compute capability >= 0.0): 0 (Note: TensorFlow was not compiled with CUDA or ROCm support)
I0000 00:00:1790594343.397562  782487 single_machine.cc:376] Starting new session


Instructions for updating:
This API was designed for TensorFlow v1. See https://www.tensorflow.org/guide/migrate for instructions on how to migrate your code to TensorFlow v2.

=========================Options=============================
Total Parameters: 2,422,210
Trainable Parameters: 164,226
MobileNetV2 FLOPs (measured): 613,054,540
-max_depth                  10000
-min_bytes                  0
-min_peak_bytes             0
-min_residual_bytes         0
-min_output_bytes           0
-min_micros                 0
-min_accelerator_micros     0
-min_cpu_micros             0
-min_params                 0
-min_float_ops              1
-min_occurrence             0
-step                       -1
-order_by                   float_ops
-account_type_regexes       .*
-start_name_regexes         .*
-trim_name_regexes          
-show_name_regexes          .*
-hide_name_regexes          
-account_displayed_op_only  true
-select                     float_ops
-output                     stdout:



===== COMPUTATIONAL COST SUMMARY =====
Model Training Time: 242.05 seconds
Memory Used During Training: -0.36 MB
Total Model Parameters: 2,422,210
Trainable Parameters: 164,226
MobileNetV2 FLOPs (measured): 613,054,540


Deep Feature Extraction

In [2]:
import time
from tensorflow.keras.models import load_model, Model
import pickle
import numpy as np
import pandas as pd
from pathlib import Path
import os
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.model_selection import train_test_split

start_time = time.time()

# BTT dataset and confirmed checkpoint
image_dir = Path('/Users/kaivalyapuranik/Downloads/archive')
model_path = '/Users/kaivalyapuranik/Desktop/XAI-Med-Images-Stat-Visual-Rules/BTTypes_MobileNetV2.keras'
feature_dir = 'extracted_features_BTTypes'
os.makedirs(feature_dir, exist_ok=True)

filepaths = list(image_dir.glob('**/*.png'))
labels = [path.parent.name for path in filepaths]
image_df = pd.DataFrame({'Filepath': [str(path) for path in filepaths], 'Label': labels})
image_df = image_df.sample(frac=1, random_state=1).reset_index(drop=True)
train_df, temp_df = train_test_split(image_df, train_size=0.7, shuffle=True, random_state=1)
val_df, test_df = train_test_split(temp_df, train_size=0.5, shuffle=True, random_state=1)


def create_gen():
    train_generator = ImageDataGenerator(
        preprocessing_function=tf.keras.applications.mobilenet_v2.preprocess_input,
        rotation_range=20,
        width_shift_range=0.2,
        height_shift_range=0.2,
        shear_range=0.2,
        zoom_range=0.2,
        horizontal_flip=True,
        fill_mode='nearest'
    )
    val_test_generator = ImageDataGenerator(
        preprocessing_function=tf.keras.applications.mobilenet_v2.preprocess_input
    )
    common_args = {
        'target_size': (224, 224),
        'color_mode': 'rgb',
        'class_mode': 'categorical',
        'batch_size': 16
    }
    train_images = train_generator.flow_from_dataframe(
        train_df, x_col='Filepath', y_col='Label', shuffle=True, seed=0, **common_args
    )
    val_images = val_test_generator.flow_from_dataframe(
        val_df, x_col='Filepath', y_col='Label', shuffle=False, **common_args
    )
    test_images = val_test_generator.flow_from_dataframe(
        test_df, x_col='Filepath', y_col='Label', shuffle=False, **common_args
    )
    return train_images, val_images, test_images

loaded_model = load_model(model_path)
feature_extractor = Model(
    inputs=loaded_model.input,
    outputs=loaded_model.layers[-4].output
)
train_images, val_images, test_images = create_gen()


def extract_features(data_gen, set_name):
    features = []
    labels = []
    data_gen.reset()
    for _ in range(len(data_gen)):
        batch_images, batch_labels = next(data_gen)
        features.append(feature_extractor.predict(batch_images, verbose=0))
        labels.append(batch_labels)
    features = np.concatenate(features, axis=0)
    labels = np.concatenate(labels, axis=0)
    with open(os.path.join(feature_dir, f'{set_name}_features.pkl'), 'wb') as file:
        pickle.dump((features, labels), file)
    print(f'{set_name}: features={features.shape}, labels={labels.shape}')


extract_features(train_images, 'train')
extract_features(val_images, 'val')
extract_features(test_images, 'test')

elapsed_time = time.time() - start_time
print(f'Total Feature Extraction Time: {elapsed_time:.2f} seconds ({elapsed_time / 60:.2f} minutes)')

Found 1680 validated image filenames belonging to 2 classes.
Found 360 validated image filenames belonging to 2 classes.
Found 360 validated image filenames belonging to 2 classes.
train: features=(1680, 128), labels=(1680, 2)
val: features=(360, 128), labels=(360, 2)
test: features=(360, 128), labels=(360, 2)
Total Feature Extraction Time: 80.78 seconds (1.35 minutes)


Statistical Feature Caculation

In [3]:
import numpy as np
import pickle
import pandas as pd
from scipy.stats import skew, entropy, kurtosis, variation, iqr
import os
import time

feature_dir = 'extracted_features_BTTypes'
stat_feature_dir = 'full_statistical_features_BTTypes'
os.makedirs(stat_feature_dir, exist_ok=True)

with open(os.path.join(feature_dir, 'train_features.pkl'), 'rb') as file:
    train_features, train_labels = pickle.load(file)
with open(os.path.join(feature_dir, 'val_features.pkl'), 'rb') as file:
    val_features, val_labels = pickle.load(file)
with open(os.path.join(feature_dir, 'test_features.pkl'), 'rb') as file:
    test_features, test_labels = pickle.load(file)


def signal_to_noise(feature_vector):
    return np.mean(feature_vector) / (np.std(feature_vector) + 1e-6)


def calculate_statistical_features(features):
    rows = []
    for feature_vector in features:
        rows.append({
            'mean': np.mean(feature_vector),
            'std_dev': np.std(feature_vector),
            'variance': np.var(feature_vector),
            'median': np.median(feature_vector),
            'range': np.ptp(feature_vector),
            'skewness': skew(feature_vector),
            'kurtosis': kurtosis(feature_vector),
            'entropy': entropy(np.abs(feature_vector) + 1e-6),
            'energy': np.sum(feature_vector ** 2),
            'contrast': np.std(feature_vector) ** 2,
            'mean_abs_dev': np.mean(np.abs(feature_vector - np.mean(feature_vector))),
            'min_value': np.min(feature_vector),
            'max_value': np.max(feature_vector),
            'iqr': iqr(feature_vector),
            'percentile_25': np.percentile(feature_vector, 25),
            'percentile_50': np.percentile(feature_vector, 50),
            'percentile_75': np.percentile(feature_vector, 75),
            'signal_to_noise': signal_to_noise(feature_vector),
            'coef_of_var': variation(feature_vector),
            'autocorrelation': np.corrcoef(feature_vector[:-1], feature_vector[1:])[0, 1],
            'shannon_entropy': -np.sum(feature_vector * np.log2(feature_vector + 1e-6)),
            'root_mean_square': np.sqrt(np.mean(feature_vector ** 2)),
            'harmonic_mean': len(feature_vector) / np.sum(1.0 / (feature_vector + 1e-6)),
            'geometric_mean': np.exp(np.mean(np.log(feature_vector + 1e-6))),
            'std_error_mean': np.std(feature_vector) / np.sqrt(len(feature_vector)),
            'median_abs_dev': np.median(np.abs(feature_vector - np.median(feature_vector)))
        })
    return pd.DataFrame(rows)


start_time = time.time()
for set_name, features, labels in [
    ('train', train_features, train_labels),
    ('val', val_features, val_labels),
    ('test', test_features, test_labels)
]:
    frame = calculate_statistical_features(features)
    frame['label'] = np.argmax(labels, axis=1) if labels.ndim > 1 and labels.shape[1] > 1 else labels
    frame.to_csv(os.path.join(stat_feature_dir, f'{set_name}_stat_features.csv'), index=False)

elapsed_time = time.time() - start_time
print(f'Total Statistical Feature Calculation Time: {elapsed_time:.2f} seconds ({elapsed_time / 60:.2f} minutes)')

Total Statistical Feature Calculation Time: 3.79 seconds (0.06 minutes)


ZFMIS Feature Selection 

In [4]:
import os
import time
import pandas as pd
from sklearn.feature_selection import SelectKBest, mutual_info_classif

input_dir = 'full_statistical_features_BTTypes'
output_dir = 'zfmis_selected_features_BTTypes'
os.makedirs(output_dir, exist_ok=True)

train_data = pd.read_csv(os.path.join(input_dir, 'train_stat_features.csv'))
val_data = pd.read_csv(os.path.join(input_dir, 'val_stat_features.csv'))
test_data = pd.read_csv(os.path.join(input_dir, 'test_stat_features.csv'))


def find_zero_features(dataframe):
    return [
        column for column in dataframe.columns
        if column != 'label' and (dataframe[column] == 0).mean() > 0.5
    ]


start_time = time.time()
zero_features = set()
for dataframe in (train_data, val_data, test_data):
    zero_features.update(find_zero_features(dataframe))

filtered_tables = [
    dataframe.drop(columns=zero_features)
    for dataframe in (train_data, val_data, test_data)
]
feature_columns = [column for column in filtered_tables[0].columns if column != 'label']

selector = SelectKBest(score_func=mutual_info_classif, k=3)
selector.fit(filtered_tables[0][feature_columns], filtered_tables[0]['label'])
selected_columns = list(pd.Index(feature_columns)[selector.get_support()])

k_output_dir = os.path.join(output_dir, 'k_3')
os.makedirs(k_output_dir, exist_ok=True)
for set_name, dataframe in zip(('train', 'val', 'test'), filtered_tables):
    dataframe[selected_columns + ['label']].to_csv(
        os.path.join(k_output_dir, f'{set_name}_selected_features.csv'),
        index=False
    )

elapsed_time = time.time() - start_time
print(f'Zero-filtered features: {sorted(zero_features)}')
print(f'Selected features: {selected_columns}')
print(f'Total ZFMIS Time: {elapsed_time:.2f} seconds ({elapsed_time / 60:.2f} minutes)')

Zero-filtered features: ['median', 'median_abs_dev', 'min_value', 'percentile_25', 'percentile_50']
Selected features: ['entropy', 'coef_of_var', 'autocorrelation']
Total ZFMIS Time: 31.29 seconds (0.52 minutes)


Decision Tree Training Time on All Statistical Feature Set  aand 
Decision Tree Testing Time All Statistical Feature Set 


In [5]:
import os
import json
import numpy as np
import pandas as pd
import time
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

stat_feature_dir = 'zfmis_selected_features_BTTypes/k_3'
with open(os.path.join(stat_feature_dir, 'best_dt_params.json'), 'r') as file:
    dt_params = json.load(file)['best_dt_params']

train_df = pd.read_csv(os.path.join(stat_feature_dir, 'train_selected_features.csv'))
val_df = pd.read_csv(os.path.join(stat_feature_dir, 'val_selected_features.csv'))
test_df = pd.read_csv(os.path.join(stat_feature_dir, 'test_selected_features.csv'))

train_stat_features = train_df.drop(columns=['label']).values
train_labels = train_df['label'].values
val_stat_features = val_df.drop(columns=['label']).values
val_labels = val_df['label'].values
test_stat_features = test_df.drop(columns=['label']).values
test_labels = test_df['label'].values

combined_features = np.vstack([train_stat_features, val_stat_features])
combined_labels = np.hstack([train_labels, val_labels])

clf = DecisionTreeClassifier(random_state=0, **dt_params)
print(f'Using Decision Tree parameters: {dt_params}')

train_start_time = time.time()
clf.fit(combined_features, combined_labels)
train_end_time = time.time()
total_train_time = train_end_time - train_start_time

train_predictions = clf.predict(combined_features)
train_accuracy = accuracy_score(combined_labels, train_predictions)
print(f'Combined Training Accuracy: {train_accuracy * 100:.4f}%')
print(f'Combined Training Confusion Matrix:\n{confusion_matrix(combined_labels, train_predictions)}')
print(classification_report(combined_labels, train_predictions))

test_start_time = time.time()
test_predictions = clf.predict(test_stat_features)
test_end_time = time.time()
total_test_time = test_end_time - test_start_time

test_accuracy = accuracy_score(test_labels, test_predictions)
print(f'Test Accuracy: {test_accuracy * 100:.4f}%')
print(f'Test Confusion Matrix:\n{confusion_matrix(test_labels, test_predictions)}')
print(classification_report(test_labels, test_predictions))
print(f'Total Training Time: {total_train_time:.4f} seconds ({total_train_time / 60:.4f} minutes)')
print(f'Total Testing Time: {total_test_time:.4f} seconds ({total_test_time / 60:.4f} minutes)')

Using Decision Tree parameters: {'criterion': 'entropy', 'max_depth': 5, 'min_samples_leaf': 4, 'min_samples_split': 2}
Combined Training Accuracy: 91.4216%
Combined Training Confusion Matrix:
[[899 129]
 [ 46 966]]
              precision    recall  f1-score   support

           0       0.95      0.87      0.91      1028
           1       0.88      0.95      0.92      1012

    accuracy                           0.91      2040
   macro avg       0.92      0.91      0.91      2040
weighted avg       0.92      0.91      0.91      2040

Test Accuracy: 89.7222%
Test Confusion Matrix:
[[148  24]
 [ 13 175]]
              precision    recall  f1-score   support

           0       0.92      0.86      0.89       172
           1       0.88      0.93      0.90       188

    accuracy                           0.90       360
   macro avg       0.90      0.90      0.90       360
weighted avg       0.90      0.90      0.90       360

Total Training Time: 0.0106 seconds (0.0002 minutes)
Total T

RuleFit Training Time All Statistical Feature Set  and
RuleFit Testing Time All Statistical Feature Set  


In [6]:
import os
import json
import pandas as pd
import numpy as np
import time
from rulefit import RuleFit
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

input_dir = 'full_statistical_features_BTTypes'
with open(os.path.join('rulefit_BTTypes', 'best_rf_params.json'), 'r') as file:
    rf_params = json.load(file)['best_rf_params']

train_data = pd.read_csv(os.path.join(input_dir, 'train_stat_features.csv'))
val_data = pd.read_csv(os.path.join(input_dir, 'val_stat_features.csv'))
test_data = pd.read_csv(os.path.join(input_dir, 'test_stat_features.csv'))

train_features = train_data.drop(columns=['label']).values
train_labels = train_data['label'].values
val_features = val_data.drop(columns=['label']).values
val_labels = val_data['label'].values
test_features = test_data.drop(columns=['label']).values
test_labels = test_data['label'].values

train_val_features = np.vstack([train_features, val_features])
train_val_labels = np.hstack([train_labels, val_labels])
feature_names = train_data.columns.drop('label').tolist()

rf = RuleFit(random_state=42, **rf_params)
print(f'Using RuleFit parameters: {rf_params}')

train_start_time = time.perf_counter()
rf.fit(train_val_features, train_val_labels, feature_names=feature_names)
train_end_time = time.perf_counter()
total_train_time = train_end_time - train_start_time

test_start_time = time.perf_counter()
test_predictions = rf.predict(test_features)
test_end_time = time.perf_counter()
total_test_time = test_end_time - test_start_time

test_predictions_discrete = np.round(test_predictions).astype(int)
test_predictions_discrete = np.clip(
    test_predictions_discrete,
    np.min(train_val_labels),
    np.max(train_val_labels)
)

test_accuracy = accuracy_score(test_labels, test_predictions_discrete)
print(f'Testing Accuracy: {test_accuracy:.4f}')
print(f'Test Confusion Matrix:\n{confusion_matrix(test_labels, test_predictions_discrete)}')
print(classification_report(test_labels, test_predictions_discrete, digits=4))

rules = rf.get_rules()
rules = rules[rules.coef != 0].sort_values('importance', ascending=False)
output_file_path = os.path.join(input_dir, 'rulefit_rules_on_statistical_features_BTT.txt')
rules.to_csv(output_file_path, index=False)
print(f'Rules saved to: {output_file_path}')
print(f'Total Training Time: {total_train_time:.8f} seconds ({total_train_time / 60:.8f} minutes)')
print(f'Total Testing Time: {total_test_time:.8f} seconds ({total_test_time / 60:.8f} minutes)')

Using RuleFit parameters: {'tree_size': 3, 'sample_fract': 0.7, 'max_rules': 2000}


/Users/kaivalyapuranik/Desktop/XAI-Med-Images-Stat-Visual-Rules/.venv-new/lib/python3.13/site-packages/sklearn/linear_model/_coordinate_descent.py:842: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 5.002678e-02, tolerance: 3.400e-02
  model = cd_fast.enet_coordinate_descent(
/Users/kaivalyapuranik/Desktop/XAI-Med-Images-Stat-Visual-Rules/.venv-new/lib/python3.13/site-packages/sklearn/linear_model/_coordinate_descent.py:842: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 7.359217e-02, tolerance: 3.400e-02
  model = cd_fast.enet_coordinate_descent(
/Users/kaivalyapuranik/Desktop/XAI-Med-Images-Stat-Visual-Rules/.venv-new/lib/python3.13/site-packages/sklearn/linear_model/_coordinate_descent.py:842: ConvergenceWarning

Testing Accuracy: 0.9667
Test Confusion Matrix:
[[165   7]
 [  5 183]]
              precision    recall  f1-score   support

           0     0.9706    0.9593    0.9649       172
           1     0.9632    0.9734    0.9683       188

    accuracy                         0.9667       360
   macro avg     0.9669    0.9664    0.9666       360
weighted avg     0.9667    0.9667    0.9667       360

Rules saved to: full_statistical_features_BTTypes/rulefit_rules_on_statistical_features_BTT.txt
Total Training Time: 95.74252904 seconds (1.59570882 minutes)
Total Testing Time: 0.00819679 seconds (0.00013661 minutes)


/Users/kaivalyapuranik/Desktop/XAI-Med-Images-Stat-Visual-Rules/.venv-new/lib/python3.13/site-packages/sklearn/linear_model/_coordinate_descent.py:842: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 1.931388e+00, tolerance: 5.100e-02
  model = cd_fast.enet_coordinate_descent(


SFMOV Time

In [7]:
import os
import time
from pathlib import Path
import cv2
import numpy as np
import matplotlib.pyplot as plt
from tensorflow.keras.models import Model, load_model
from tensorflow.keras.preprocessing.image import load_img, img_to_array
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input
from scipy.stats import skew
from skimage.measure import shannon_entropy

model_path = '/Users/kaivalyapuranik/Desktop/XAI-Med-Images-Stat-Visual-Rules/BTTypes_MobileNetV2.keras'
image_dir = Path('/Users/kaivalyapuranik/Downloads/archive')
output_dir = Path('/Users/kaivalyapuranik/Desktop/XAI-Med-Images-Stat-Visual-Rules/heatmap_overlays_BTTypes/timed')
output_dir.mkdir(parents=True, exist_ok=True)

loaded_model = load_model(model_path)
conv_layer_model = Model(
    inputs=loaded_model.input,
    outputs=loaded_model.get_layer('Conv_1_bn').output
)
dense_layer_model = Model(
    inputs=loaded_model.input,
    outputs=loaded_model.get_layer('dense_1').output
)

sample_image = next(
    path for class_dir in sorted(image_dir.iterdir()) if class_dir.is_dir()
    for path in sorted(class_dir.iterdir())
    if path.suffix.lower() in {'.png', '.jpg', '.jpeg'}
)
start_time = time.time()
input_size = (224, 224)
original_image = load_img(sample_image, target_size=input_size)
image_array = img_to_array(original_image)
model_input = preprocess_input(np.expand_dims(image_array.copy(), axis=0))
conv_features = conv_layer_model.predict(model_input, verbose=0).squeeze()
dense_features = dense_layer_model.predict(model_input, verbose=0).squeeze()

mean_map = np.zeros(conv_features.shape[:2])
skewness_map = np.zeros(conv_features.shape[:2])
entropy_map = np.zeros(conv_features.shape[:2])
for channel_index in range(conv_features.shape[-1]):
    feature_map = conv_features[:, :, channel_index]
    mean_map += feature_map / conv_features.shape[-1]
    skewness_map += skew(feature_map.ravel()) * feature_map
    entropy_map += shannon_entropy(feature_map) * feature_map

normalize = lambda values: (values - values.min()) / (values.max() - values.min() + 1e-8)
combined_map = (
    np.mean(dense_features) * normalize(mean_map)
    + skew(dense_features) * normalize(skewness_map)
    + shannon_entropy(dense_features) * normalize(entropy_map)
)
figure, axes = plt.subplots(1, 2, figsize=(10, 5))
axes[0].imshow(original_image)
axes[0].set_title('Original BTT Image')
axes[1].imshow(original_image)
axes[1].imshow(cv2.resize(normalize(combined_map), input_size), cmap='jet', alpha=0.5)
axes[1].set_title('Dense-weighted SFMOV Heatmap')
for axis in axes:
    axis.axis('off')
figure.tight_layout()
output_path = output_dir / f'{sample_image.stem}_timed_heatmap.png'
figure.savefig(output_path, dpi=150, bbox_inches='tight')
plt.close(figure)
elapsed_time = time.time() - start_time
print(f'SFMOV overlay saved to: {output_path}')
print(f'Total SFMOV Time: {elapsed_time:.2f} seconds ({elapsed_time / 60:.2f} minutes)')

SFMOV overlay saved to: /Users/kaivalyapuranik/Desktop/XAI-Med-Images-Stat-Visual-Rules/heatmap_overlays_BTTypes/timed/B (1)_timed_heatmap.png
Total SFMOV Time: 31.11 seconds (0.52 minutes)


RuleFit rule extraction Code 

In [9]:
import json
import os
import time
import pandas as pd
import numpy as np
from rulefit import RuleFit

input_dir = 'full_statistical_features_BTTypes'
with open(os.path.join('rulefit_BTTypes', 'best_rf_params.json'), 'r') as file:
    rf_params = json.load(file)['best_rf_params']

train_df = pd.read_csv(os.path.join(input_dir, 'train_stat_features.csv'))
val_df = pd.read_csv(os.path.join(input_dir, 'val_stat_features.csv'))
train_val_df = pd.concat([train_df, val_df], ignore_index=True)
X_train_val = train_val_df.drop(columns=['label']).values
y_train_val = train_val_df['label'].values
feature_names = train_val_df.columns.drop('label').tolist()

start_time = time.time()
rulefit_model = RuleFit(random_state=42, **rf_params)
rulefit_model.fit(X_train_val, y_train_val, feature_names=feature_names)
rules = rulefit_model.get_rules()
rules = rules[rules.coef != 0]

if not rules.empty:
    rules['rule'] = rules['rule'].astype(str)
    rules_path = os.path.join(input_dir, 'simplified_rulefit_top_if_then_rules_for_all_classes.txt')
    with open(rules_path, 'w') as file:
        for _, row in rules.sort_values('importance', ascending=False).head(40).iterrows():
            file.write(
                f"If ({row['rule']}) then RuleFit output "
                f"(Importance: {row['importance']:.4f})\n"
            )
    print(f'RuleFit rules saved to: {rules_path}')

elapsed_time = time.time() - start_time
print(f'Total RuleFit rule extraction time: {elapsed_time:.2f} seconds ({elapsed_time / 60:.2f} minutes)')

/Users/kaivalyapuranik/Desktop/XAI-Med-Images-Stat-Visual-Rules/.venv-new/lib/python3.13/site-packages/sklearn/linear_model/_coordinate_descent.py:842: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 5.002678e-02, tolerance: 3.400e-02
  model = cd_fast.enet_coordinate_descent(
/Users/kaivalyapuranik/Desktop/XAI-Med-Images-Stat-Visual-Rules/.venv-new/lib/python3.13/site-packages/sklearn/linear_model/_coordinate_descent.py:842: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 7.359217e-02, tolerance: 3.400e-02
  model = cd_fast.enet_coordinate_descent(
/Users/kaivalyapuranik/Desktop/XAI-Med-Images-Stat-Visual-Rules/.venv-new/lib/python3.13/site-packages/sklearn/linear_model/_coordinate_descent.py:842: ConvergenceWarning

RuleFit rules saved to: full_statistical_features_BTTypes/simplified_rulefit_top_if_then_rules_for_all_classes.txt
Total RuleFit rule extraction time: 93.88 seconds (1.56 minutes)


/Users/kaivalyapuranik/Desktop/XAI-Med-Images-Stat-Visual-Rules/.venv-new/lib/python3.13/site-packages/sklearn/linear_model/_coordinate_descent.py:842: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 1.931388e+00, tolerance: 5.100e-02
  model = cd_fast.enet_coordinate_descent(


Decision tree rules 

In [10]:
import json
import os
import numpy as np
import pandas as pd
import time
from sklearn.tree import DecisionTreeClassifier

stat_feature_dir = 'zfmis_selected_features_BTTypes/k_3'
with open(os.path.join(stat_feature_dir, 'best_dt_params.json'), 'r') as file:
    dt_params = json.load(file)['best_dt_params']

train_df = pd.read_csv(os.path.join(stat_feature_dir, 'train_selected_features.csv'))
val_df = pd.read_csv(os.path.join(stat_feature_dir, 'val_selected_features.csv'))
train_df = pd.concat([train_df, val_df], ignore_index=True)
combined_features = train_df.drop(columns=['label']).values
combined_labels = train_df['label'].values
feature_names = train_df.columns.drop('label').tolist()

start_time = time.time()
clf = DecisionTreeClassifier(random_state=0, **dt_params)
clf.fit(combined_features, combined_labels)
rules = []


def traverse_tree(tree, node=0, conditions='', path_importance=0):
    if tree.children_left[node] == -1 and tree.children_right[node] == -1:
        class_value = np.argmax(tree.value[node][0])
        rules.append(
            f'If ({conditions.rstrip(" & ")}) then Class = {class_value} '
            f'(Importance: {path_importance:.4f})'
        )
        return
    feature_index = tree.feature[node]
    threshold = tree.threshold[node]
    if feature_index >= 0:
        updated_importance = path_importance + clf.feature_importances_[feature_index]
        traverse_tree(
            tree,
            tree.children_left[node],
            conditions + f'{feature_names[feature_index]} <= {threshold:.3f} & ',
            updated_importance
        )
        traverse_tree(
            tree,
            tree.children_right[node],
            conditions + f'{feature_names[feature_index]} > {threshold:.3f} & ',
            updated_importance
        )


traverse_tree(clf.tree_)
rules.sort(key=lambda rule: float(rule.split('(Importance: ')[1].split(')')[0]), reverse=True)
output_file_path = os.path.join(stat_feature_dir, 'simplified_sorted_rules.txt')
with open(output_file_path, 'w') as file:
    for rule in rules:
        file.write(rule + '\n')

elapsed_time = time.time() - start_time
print(f'Decision Tree rules saved to: {output_file_path}')
print(f'Total Decision Tree training and rule extraction time: {elapsed_time:.4f} seconds')

Decision Tree rules saved to: zfmis_selected_features_BTTypes/k_3/simplified_sorted_rules.txt
Total Decision Tree training and rule extraction time: 1.1201 seconds
